# 數位控制系統第二章：離散時間系統與 z 轉換（教學版 Notebook）

本 Notebook 是 `chp2.md` 教材的教學版，額外補充：
- 每個第一次出現的 MATLAB / Octave 函數（`ztrans`, `iztrans`, `residue`, `conv`, `eig`, `ss2tf`, ...）的逐步解說
- 每個公式的推導過程與符號定義
- 公式 → 程式碼的逐項對照
- 可直接執行的程式碼範例

建議搭配 `chp2.md`（完整理論與推導）與 `chp2.m`（精簡可執行版）一起閱讀。

---

## 🚀 開始前必讀（Octave 使用者）

本 Notebook 需要**同資料夾**的三個輔助檔，用來補上 Octave 相對於 MATLAB 缺少的功能：

| 檔案 | 補什麼 |
|---|---|
| `iztrans.m` | 反 z 轉換。Octave 的 symbolic 套件（到 3.2.2）**沒有實作** `iztrans` |
| `ztrans_cf.m` | z 轉換的**閉合形式**。套件的 `ztrans` 常常只回傳未求和的 `Sum` |
| `sym_magic.py` | 用 **LaTeX** 顯示符號運算式（`%sym` / `%%sym`） |

完整說明見同資料夾的 **`TOOLS.md`**（含環境安裝、函式參考、踩過的坑）。

### 執行順序

1. 先跑下面那格 `%%python` —— 註冊 LaTeX magic，**每次重啟 kernel 都要重跑**
2. 再跑「環境設定」那格 —— 載入 Octave 套件
3. 之後由上而下依序執行

### `%sym` / `%%sym` 用法

Octave kernel 只會把符號運算式當**純文字**送出（用 `/ \ -` 拼出來的排版）。
要漂亮的數學式就用這兩個 magic：

| 寫法 | 說明 |
|---|---|
| `%%sym` | cell magic。整格只放變數名，一行一個 |
| `%sym Ez` | line magic。但**必須是 cell 的第一行** |

> ⚠️ **magic 只能放在 cell 開頭**，不能混在 Octave 程式碼中間 —— metakernel 只解析
> cell 開頭的 magic。所以本 Notebook 的符號運算都拆成「**計算格**」＋「**顯示格**」兩格，
> 計算格結尾用 `;` 抑制純文字輸出。

> ⚠️ **LaTeX 只是換排版，不會讓算不出來的東西變成算得出來。**
> `ztrans(1^k)` 回傳的是**分段函數**（SymPy 判斷不出收斂性），渲染成 LaTeX
> 仍然是一大坨 `\begin{cases}`；要得到 $\dfrac{z}{z-1}$ 必須改用 `ztrans_cf`。

### 不想用 LaTeX 的話

環境設定格裡已經設了 `sympref('display','unicode')`，純文字排版會比預設的
`ascii` 模式好看。也可以改成 `'flat'`（單行，方便複製）。

### 在 MATLAB 執行的話

把 `pkg load ...`、`%%python`、`%sym` / `%%sym` 這些都刪掉。MATLAB 內建
`ztrans` / `iztrans`，只要把 `ztrans_cf(f, k, z)` 改回 `ztrans(f, k, z)` 即可。

---


In [14]:
%%python
# 註冊 LaTeX 顯示 magic（%sym / %%sym）。每次重啟 kernel 都要重跑這格。
#
# kernel 的工作目錄就是本 notebook 所在資料夾，所以 os.getcwd() 剛好指到
# sym_magic.py 旁邊 —— 不需要任何絕對路徑，專案搬到別台機器一樣能用。
import os, sys
sys.path.insert(0, os.getcwd())
import sym_magic
sym_magic.register_magics(kernel)
print('LaTeX magic 已就緒：用 %sym <變數> 或 %%sym 顯示符號運算式')


LaTeX magic 已就緒：用 %sym <變數> 或 %%sym 顯示符號運算式


## 🔧 環境設定

在 Octave Jupyter Notebook 中繪圖與使用符號運算工具箱，需要：
1. 使用 `graphics_toolkit('gnuplot')` 讓圖表能內嵌顯示
2. 載入 `control`（`tf`, `ss`）、`symbolic`（`syms`, `ztrans`）、`signal`（`ss2tf`）三個套件
   —— ⚠️ `ss2tf` 在 Octave 屬於 **`signal`** 套件，只載入 `control` 會出現 `error: 'ss2tf' undefined`
3. 設定中文字型以避免亂碼

> 若你是在標準 MATLAB（非 Octave）環境執行，`pkg load ...` 這幾行會報錯，直接刪除或註解掉即可 —— MATLAB 內建 Control System Toolbox 與 Symbolic Math Toolbox，不需要額外載入套件。


In [15]:
%plot --format svg
% 上面這行是 Octave Jupyter kernel 的「magic」語法（必須放在 cell 第一行），
% 讓圖表以 SVG 格式內嵌顯示，SVG 對中文字型的支援較好，可避免圖表中的中文標題/座標軸文字亂碼。

warning('off', 'Octave:graphics-toolkit');
graphics_toolkit('gnuplot');
clear; clc;
pkg load control;    % 若在 MATLAB 中執行，請刪除或註解此行
pkg load symbolic;   % 若在 MATLAB 中執行，請刪除或註解此行
pkg load signal;     % ss2tf 在 Octave 屬於 signal 套件（MATLAB 放在 Control System Toolbox）

sympref('display', 'unicode');   % 符號輸出改用 unicode 排版，比預設的 ascii 好看

set(0, 'DefaultTextFontName', 'Microsoft JhengHei');
set(0, 'DefaultAxesFontName', 'Microsoft JhengHei');


## 一、離散時間系統的基本概念 (2.1–2.2)

**連續時間系統**由微分方程式描述；**離散時間系統**由**差分方程式**（difference equation）描述——因為數位電腦只能在固定取樣瞬間 $kT$（$k=0,1,2,\dots$）讀取與輸出數值。

一個典型的例子：把類比 PI 控制器 $m(t)=K_Pe(t)+K_I\int_0^t e(\tau)d\tau$ 數位化。用矩形法則數值積分近似積分項：

$$
x(kT) = x[(k-1)T] + Te(kT)
$$

這是一個**一階差分方程式**。推廣到 $n$ 階線性非時變差分方程式的一般形式：

$$
x(k) = b_n e(k) + b_{n-1}e(k-1) + \cdots + b_0 e(k-n) - a_{n-1}x(k-1) - \cdots - a_0 x(k-n)
$$

**符號定義**：

| 符號 | 意義 |
|---|---|
| $T$ | 取樣週期（秒），即取樣瞬間之間的時間間隔 |
| $k$ | 取樣時刻的整數編號，實際時間為 $kT$ |
| $a_i, b_i$ | 差分方程式（數位濾波器）的係數 |
| $n$ | 差分方程式的階數 |

> **觀念解析**：把 $T$ 決定得夠小、把差分方程式的係數設計得當，數位濾波器的行為就能逼近原本的類比濾波器——這正是「用數位電腦取代類比電路」的數學基礎。

## 二、z 轉換的定義 (2.3)

正如拉普拉斯轉換把微分方程式變成代數方程式，**z 轉換**把差分方程式變成代數方程式。z 轉換是對**數列** $\{e(k)\}$ 定義的：

$$
E(z) = \mathcal{Z}[\{e(k)\}] = \sum_{k=0}^{\infty} e(k)z^{-k}
$$

也就是說，$E(z)$ 是變數 $z^{-1}$ 的冪級數，**級數的每一項係數就是數列 $e(k)$ 的值**。這個「係數＝數列值」的性質，是整章 z 轉換理論的出發點。

### 公式 → 程式碼：`ztrans()`

MATLAB/Octave 的 `ztrans(expr)` 對符號表達式（以 `k` 為自變數的數列）直接計算 z 轉換，等同於自動完成上面的無窮級數求和。使用前要先用 `syms` 宣告符號變數。

In [16]:
syms k a T z

% 例 2.2：e(k)=1（所有 k），求 E(z)
Ez_step = ztrans(1^k)
% 理論結果：E(z) = z/(z-1)


Ez_step = (sym)

  ⎧    1           1     
  ⎪  ─────    for ─── < 1
  ⎪      1        │z│    
  ⎪  1 - ─               
  ⎪      z               
  ⎪                      
  ⎪  ∞                   
  ⎨ ___                  
  ⎪ ╲                    
  ⎪  ╲    -n             
  ⎪  ╱   z     otherwise 
  ⎪ ╱                    
  ⎪ ‾‾‾                  
  ⎪n = 0                 
  ⎩                      



> ### ⚠️ 上面那格的輸出不是 $\dfrac{z}{z-1}$
>
> Octave 的 `ztrans` 直接把定義式 $E(z)=\sum_{k=0}^{\infty}e(kT)z^{-k}$ 丟給 SymPy 求和。
> SymPy 判斷不出 $\left|\frac{1}{z}\right|<1$ 是否成立，所以回傳一個**分段函數**：
> 收斂時是 $\frac{1}{1-1/z}$，否則保留未求和的 $\sum$。
>
> **這無法用 `assume` 解決** —— 收斂條件是「兩個自由符號之間的不等式」，
> 不在 SymPy 假設系統的表達能力內。
>
> 同資料夾的 `ztrans_cf.m` 改用替換法：把 $底數^{ck}$ 的底數換成單一啞符號 $q$，
> SymPy 對 $\sum q^k z^{-k}$ 有現成的幾何級數公式，取其收斂支後再把 $q$ 代回。
> 等於是**宣告自己位在收斂域（ROC）內** —— 跟課本查 z 轉換表的前提相同。
>
> 下一格用 `ztrans_cf` 重算一次，就會得到教科書的形式。
> （在 MATLAB 則不需要這一步，內建的 `ztrans` 直接就會給閉合形式。）


In [17]:
% 用 ztrans_cf 重算：取收斂域內的閉合形式
Ez_step_cf = ztrans_cf(sym(1)^k, k, z);   % 1^k 會化簡成常數 1，所以要明確指定 k、z


In [18]:
%%sym
Ez_step_cf


<IPython.core.display.Math object>

**函數介紹：`syms`**——宣告接下來出現的變數為「符號變數」，讓 MATLAB 做符號（代數）運算而非數值運算，是 Symbolic Math Toolbox 的起手式。

**函數介紹：`ztrans(expr)`**——對符號表達式 `expr`（以 `k` 為自變數）計算單邊 z 轉換 $E(z)=\sum_{k=0}^\infty e(k)z^{-k}$，回傳值以 `z` 表示。

In [19]:
% 例 2.3：e(k) = a^(kT)，求 E(z)
Ez_exp = ztrans(exp(-a*k*T))
% 理論結果：E(z) = z/(z - exp(-a*T))，因為 a^T = exp(T*ln(a)) = exp(-a*T)（此處取 e(k)=exp(-a*kT) 為例）


Ez_exp = (sym)

    ∞              
   ___             
   ╲               
    ╲    -k  -T⋅a⋅k
    ╱   z  ⋅ℯ      
   ╱               
   ‾‾‾             
  k = 0            



In [20]:
% 用 ztrans_cf 重算
Ez_exp_cf = ztrans_cf(exp(-a*k*T), k, z);


In [21]:
%%sym
Ez_exp_cf


<IPython.core.display.Math object>

**數學 ↔ MATLAB 對照**：

- 數學上的封閉形式 $E(z)=\dfrac{z}{z-a^T}$（由等比級數公式 $\frac{1}{1-x}=1+x+x^2+\cdots$，取 $x=a^Tz^{-1}$ 得到）
- ↔ MATLAB 輸出 `z/(z - exp(-a*T))`（符號引擎習慣用指數形式 $e^{-aT}$ 表示 $a^T$）

---

## 三、z 轉換的性質 (2.4)

### 1. 線性、實數平移、複數平移

| 性質 | 公式 |
|---|---|
| 線性 | $\mathcal{Z}[\alpha e_1(k)+\beta e_2(k)] = \alpha E_1(z)+\beta E_2(z)$ |
| 右移（延遲） | $\mathcal{Z}[e(k-n)u(k-n)] = z^{-n}E(z)$ |
| 左移（超前） | $\mathcal{Z}[e(k+n)u(k)] = z^n\left[E(z)-\sum_{k=0}^{n-1}e(k)z^{-k}\right]$ |
| 複數平移 | $\mathcal{Z}[a^{kT}e(kT)] = E(z)\big\vert_{z\to z/a^T}$ |

**物理意義**：右移不遺失資訊所以形式簡單；左移會把最前面 $n$ 個值「甩出視窗」，公式中要扣掉這些即將消失的項來補償。複數平移則是「時域乘上指數」對應「z 域座標縮放」。

### 2. 初值定理與終值定理

$$
e(0) = \lim_{z\to\infty}E(z), \qquad \lim_{n\to\infty}e(n) = \lim_{z\to1}(z-1)E(z)
$$

終值定理成立的條件：$E(z)$ 所有極點在單位圓內，最多允許在 $z=1$ 處有一個單重極點（第 7 章會證明）。

In [22]:
% 例 2.6：複數平移定理 -> kT * a^(kT)
ekT = (k*T)*exp(a*k*T);
Ez_complex_shift = ztrans(ekT)
pretty(Ez_complex_shift)
% 理論結果：(T*z*exp(T*a)) / (z - exp(T*a))^2


Ez_complex_shift = (sym)

    ∞                 
   ___                
   ╲                  
    ╲        -k  T⋅a⋅k
    ╱   T⋅k⋅z  ⋅ℯ     
   ╱                  
   ‾‾‾                
  k = 0               

    ∞                 
   ___                
   ╲                  
    ╲        -k  T⋅a⋅k
    ╱   T⋅k⋅z  ⋅ℯ     
   ╱                  
   ‾‾‾                
  k = 0               


In [23]:
% 用 ztrans_cf 重算
Ez_complex_shift_cf = ztrans_cf(ekT, k, z);


In [24]:
%%sym
Ez_complex_shift_cf


<IPython.core.display.Math object>

**函數介紹：`pretty(expr)`**——把符號表達式排版成較接近手寫數學的形式（分數用橫線顯示），純粹是顯示用途，方便閱讀複雜結果，不影響運算。

---

## 四、由 s 域函數求 z 轉換 (2.5) —— 例 2.9

當系統函數以 $s$ 域（連續時間）給出時，標準做法是：**先做部分分式展開，再逐項查表轉成 z 域**。

求 $E(s)=\dfrac{s^2+4s+3}{s^3+6s^2+8s}=\dfrac{s^2+4s+3}{s(s+2)(s+4)}$ 的 z 轉換。

部分分式展開（用留數法 $K_i=(s-p_i)E(s)|_{s=p_i}$）：

$$
E(s) = \frac{0.375}{s}+\frac{0.25}{s+2}+\frac{0.375}{s+4}
$$

> ⚠️ 原書 OCR 文字把中間係數印成 0.025，經重新驗算應為 **0.25**，此處已訂正。

逐項用「$\frac{1}{s+a}$ 取樣後對應 $\frac{z}{z-e^{-aT}}$」查表轉成 z 域，再相加通分即得 $E(z)$。

In [25]:
% 方法一：手動走一遍 residue -> z 域極點 -> 合併多項式
T = 0.1;
num = [1 4 3];
denom = [1 6 8 0];              % s(s+2)(s+4)，注意不可有重根
n = length(denom);
Es = tf(num, denom)              % 先建立 s 域轉移函數方便檢查


Transfer function 'Es' from input 'u1' to output ...

        s^2 + 4 s + 3  
 y1:  -----------------
      s^3 + 6 s^2 + 8 s

Continuous-time model.


In [26]:
%%sym
Es


<IPython.core.display.Math object>

**函數介紹：`tf(num, den)`**——建立連續時間轉移函數模型，`num`、`den` 為多項式係數向量（由高次到低次）。

In [27]:
[r, p, kdir] = residue(num, denom);   % 部分分式展開：留數 r、極點 p、直接項 kdir
r, p, kdir

r =

   0.3750
   0.2500
   0.3750

p =

  -4
  -2
   0

kdir = [](0x0)


In [29]:
%%sym
r
p
kdir 

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

**函數介紹：`[r,p,k] = residue(num, den)`**——對有理多項式 $\dfrac{\text{num}(s)}{\text{den}(s)}$ 做部分分式展開，回傳：
- `r`：留數（residues），對應每個極點的分子係數
- `p`：極點（poles）
- `k`：若分子階數 $\ge$ 分母階數時的直接項（多項式除法的商）

這就是手算部分分式「求 $K_0,K_1,K_2$」步驟的自動化版本。

In [36]:
pz = zeros(1, n-1);
for i = 1:n-1
    pz(i) = exp(p(i)*T);        % s 域極點 p 對應 z 域極點 exp(p*T)（複數平移概念的體現）
end

[numzz, denomz] = residue(r, pz, kdir);  % 反向呼叫 residue：用「留數+極點」合併回多項式
numz = conv(numzz, [1 0]);               % 乘上一個 z（因為查表結果都帶一個 z，如 z/(z-a)） [1 0] = 1*z^1+ 0*z^0
Ez_from_s = tf(numz, denomz, T)
% 理論結果：(z^3-1.658z^2+0.6804z)/(z^3-2.489z^2+2.038z-0.5488)


Transfer function 'Ez_from_s' from input 'u1' to output ...

          z^3 - 1.658 z^2 + 0.6804 z    
 y1:  ----------------------------------
      z^3 - 2.489 z^2 + 2.038 z - 0.5488

Sampling time: 0.1 s
Discrete-time model.


In [34]:
%%sym
pz
numzz
numz
Ez_from_s


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

**函數介紹：`residue(r, p, k)`**（反向呼叫）——已知留數與極點，回推合併成一個多項式除以多項式的有理函數，等同於「把部分分式通分」。

**函數介紹：`conv(a, b)`**——計算兩個多項式係數向量的摺積，數學上等同於多項式相乘。這裡用來把多項式乘上 $z$（`[1 0]` 代表多項式 $z$）。

**函數介紹：`tf(num, den, T)`**——多了取樣週期 `T` 參數時，建立的是**離散時間**轉移函數（自變數為 $z$）。

**數學 ↔ MATLAB 對照**：

| 數學 | MATLAB |
|---|---|
| $E(s)$ 部分分式展開，求 $K_0,K_1,K_2$ | `[r,p,k]=residue(num,denom)` |
| $\frac{1}{s+a}\to\frac{z}{z-e^{-aT}}$（逐項查表） | `pz(i)=exp(p(i)*T)` |
| 通分合併成單一有理式 | `residue(r,pz,kdir)` + `conv(numzz,[1 0])` |

In [41]:
% 方法二：先反拉普拉斯回到時域，代入 t=kT，再取 z 轉換
syms s t
Es_sym = (s^2+4*s+3)/(s^3+6*s^2+8*s);
et = ilaplace(Es_sym);          % 反拉普拉斯轉換：s 域 -> 時域 e(t)
ekT2 = subs(et, t, k*T);        % 代入 t = kT，得到取樣後的數列
Ez_alt = ztrans(ekT2);          % 對數列取 z 轉換
pretty(Ez_alt)


    double_to_sym_heuristic at line 50 column 7
    sym at line 384 column 11
    mtimes at line 54 column 3

                                  ⎛⎧                      -4    ⎞   ⎧            ↪
                                  ⎜⎪       1             ℯ      ⎟   ⎪       1    ↪
    ⎛⎧    1           1     ⎞     ⎜⎪    ───────      for ─── < 1⎟   ⎪    ─────── ↪
    ⎜⎪  ─────    for ─── < 1⎟     ⎜⎪         -4          │z│    ⎟   ⎪         -2 ↪
    ⎜⎪      1        │z│    ⎟     ⎜⎪        ℯ                   ⎟   ⎪        ℯ   ↪
    ⎜⎪  1 - ─               ⎟     ⎜⎪    1 - ───                 ⎟   ⎪    1 - ─── ↪
    ⎜⎪      z               ⎟     ⎜⎪         z                  ⎟   ⎪         z  ↪
    ⎜⎪                      ⎟     ⎜⎪                            ⎟   ⎪            ↪
    ⎜⎪  ∞                   ⎟   3⋅⎜⎨  ∞                         ⎟   ⎨  ∞         ↪
  3⋅⎜⎨ ___                  ⎟     ⎜⎪ ___                        ⎟   ⎪ ___        ↪
    ⎜⎪ ╲                    ⎟     ⎜⎪ ╲                      

In [42]:
%%sym
Ez_alt

<IPython.core.display.Math object>

> ### ⚠️ 上面那格的結果是錯的 —— 兩個 Octave 專有的陷阱
>
> 仔細看輸出，裡面**還留著 `t`**。$E(z)$ 不該有 $t$，代表 `subs` 根本沒生效。
>
> **陷阱一：`ilaplace(F)` 單引數版本。**
> 它會自己造一個**帶假設**（positive）的 `t`，跟你用 `syms s t` 宣告的 `t`
> **不是同一個符號**，所以下一行的 `subs(et, t, k*T)` 完全比對不到。
> 而且**不會報錯**，只是結果裡莫名其妙留著 `t`。
> 解法：明確寫成 `ilaplace(Es_sym, s, t)`。
>
> **陷阱二：`T = 0.1` 是 double。**
> 拿浮點數去做符號運算會觸發 `passing floating-point values to sym is dangerous`
> 警告。解法：改用 `Ts = sym(1)/10`。
>
> 再加上 `ztrans` 一樣要換成 `ztrans_cf`。下一格是修正版，並自動比對兩種解法的係數。


In [43]:
% 修正版：ilaplace 指定三個引數、T 用符號、ztrans 換成 ztrans_cf
Ts = sym(1)/10;                      % 符號 1/10，不是 double 0.1
et_fix   = ilaplace(Es_sym, s, t);   % 明確指定 s、t
ekT2_fix = subs(et_fix, t, k*Ts);    % 這次 subs 才真的生效
Ez_alt_fix = ztrans_cf(ekT2_fix, k, z);

% 驗證：正規化後的係數應與方法一的 Ez_from_s 完全相同
[na, da] = numden(Ez_alt_fix);
disp('分子係數：'); ca = double(coeffs(expand(na), z, 'all')); disp(ca / ca(1));
disp('分母係數：'); cb = double(coeffs(expand(da), z, 'all')); disp(cb / cb(1));


Waiting..........
分子係數：
   1.0000  -1.6580   0.6804        0
分母係數：
   1.0000  -2.4891   2.0379  -0.5488


In [46]:
%%sym
Ez_alt_fix
vpa(Ez_alt_fix, 5)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

**函數介紹：`ilaplace(expr, s, t)`**——對符號表達式做反拉普拉斯轉換，把 $s$ 域函數轉回時域函數 $e(t)$。

> ⚠️ **在 Octave 一定要寫成三引數 `ilaplace(F, s, t)`。** 單引數的 `ilaplace(F)` 會自己造一個
> **帶假設**（positive）的 `t`，跟你用 `syms t` 宣告的 `t` **不是同一個符號**，
> 導致下一行的 `subs(et, t, k*T)` 完全無效 —— 而且不會報錯，只是結果裡莫名其妙還留著 `t`。

**函數介紹：`subs(expr, old, new)`**——把符號表達式中的變數 `old` 代換成 `new`。這裡把時間 `t` 換成 `k*T`，對應「每隔 $T$ 秒取一個樣本」的數學操作。

**函數介紹：`vpa(expr, n)`**——**V**ariable **P**recision **A**rithmetic，把精確的符號值算成 $n$ 位有效數字的小數。

```
exp(-1/5)          ← 精確符號，沒有小數
   ↓ vpa(e, 5)
0.81873            ← 5 位有效數字
   ↓ vpa(e, 40)
0.8187307530779818586699355086190394243586
```

省略位數時預設 32 位。跟 `double()` 的差別：

| | `vpa(e, n)` | `double(e)` |
|---|---|---|
| 位數 | 任意 | 固定 ~16 位（IEEE 754 上限） |
| 回傳型別 | 仍是 `sym` | `double` |
| 能否繼續符號運算 | 可以 | 不行，已離開符號世界 |

這裡的 `vpa(Ez_alt_fix, 5)` 純粹是為了**顯示**——把 $e^{-1/5}$ 這種精確形式換算成小數，好跟方法一的 `Ez_from_s` 並排比對。兩者本來就相等，只是一個保留精確符號、一個在計算過程中就取了數值。

> ⚠️ `vpa` 餵 double 沒有意義，精度在你打字的那一刻就丟掉了：
>
> ```matlab
> vpa(0.1, 30)         % 0.100000000000000005551115123126  ← double 的誤差原形畢露
> vpa(sym(1)/10, 30)   % 0.1                               ← 這才是真的 1/10
> ```
>
> 要高精度就得**從一開始就用符號**。這也是上面寫 `Ts = sym(1)/10` 而不是 `T = 0.1` 的原因。

兩種方法（部分分式查表 vs. 先反拉普拉斯再取樣）殊途同歸，結果一致。

---


### 方法三：由 $E(z)$ 寫出差分方程式，逐次代入

前兩種方法都在求 $E(z)$ 的**閉合形式**。但如果你只想要數列本身 $e(0), e(T), e(2T), \dots$，其實不需要閉合形式 —— 把 $E(z)$ 的係數直接翻譯成差分方程式，用**例 2.10 的逐次代入法**就能一路算下去。

**推導**：把

$$E(z) = \frac{z^3 - 1.658z^2 + 0.6804z}{z^3 - 2.489z^2 + 2.038z - 0.5488}$$

分子分母同除以 $z^3$，改用 $z^{-1}$ 表示：

$$E(z) = \frac{1 - 1.658z^{-1} + 0.6804z^{-2}}{1 - 2.489z^{-1} + 2.038z^{-2} - 0.5488z^{-3}}$$

交叉相乘：

$$\left(1 - 2.489z^{-1} + 2.038z^{-2} - 0.5488z^{-3}\right)E(z) = 1 - 1.658z^{-1} + 0.6804z^{-2}$$

用**實數平移定理** $z^{-i}E(z) \leftrightarrow e(k-i)$ 取反 z 轉換（右邊的常數對應脈衝 $\delta(k-i)$）：

$$e(k) - 2.489\,e(k-1) + 2.038\,e(k-2) - 0.5488\,e(k-3) = \delta(k) - 1.658\,\delta(k-1) + 0.6804\,\delta(k-2)$$

移項，就是可以直接丟進迴圈的遞迴式：

$$e(k) = 2.489\,e(k-1) - 2.038\,e(k-2) + 0.5488\,e(k-3) \;+\; \delta(k) - 1.658\,\delta(k-1) + 0.6804\,\delta(k-2)$$

初始條件 $e(-1)=e(-2)=e(-3)=0$ —— 因果訊號，$k<0$ 時一律為零。

手算前兩項對一下：

- $e(0) = 0 - 0 + 0 \;+\; 1 = 1$
- $e(1) = 2.489(1) - 0 + 0 \;+\; 0 - 1.658 = 0.831$

> **跟例 2.10 的差別**：那題的輸入 $e(k)$ 是題目給的方波，要解的是 $m(k)$；
> 這裡的「輸入」是單位脈衝 $\delta(k)$（來自分子的常數項），輸出就是我們要的 $e(kT)$ 本身。
>
> $\delta(k)$ 定義為 $k=0$ 時等於 1、其餘為 0。


In [ ]:
%% 方法三：由 E(z) 的係數寫出差分方程式，逐次代入
[nz, dz] = tfdata(Ez_from_s, 'v');   % 取出分子、分母係數（高次 -> 低次）
nz = nz / dz(1);                     % 正規化，讓 z^3 的係數為 1
dz = dz / dz(1);

N = 10;
e_rec = zeros(1, N+1);               % e_rec(k+1) 存 e(k)（Octave 索引從 1 開始）
for kk = 0:N
    acc = 0;
    for i = 0:3                      % 右邊的脈衝項： nz(i+1) * delta(k-i)
        if kk == i
            acc = acc + nz(i+1);
        end
    end
    for i = 1:3                      % 左邊移項過來： -dz(i+1) * e(k-i)
        if kk - i >= 0
            acc = acc - dz(i+1) * e_rec(kk-i+1);
        end
    end
    e_rec(kk+1) = acc;
end

% 對照解析解：E(s) 部分分式 -> e(t) = 3/8 + (1/4)e^(-2t) + (3/8)e^(-4t)，取 t = kT
kv = 0:N;
e_exact = 3/8 + (1/4)*exp(-2*kv*T) + (3/8)*exp(-4*kv*T);

printf('   k    差分方程式       解析解 e(kT)      誤差\n');
for kk = 0:N
    printf('  %2d   %12.6f    %12.6f    %.1e\n', ...
           kk, e_rec(kk+1), e_exact(kk+1), abs(e_rec(kk+1) - e_exact(kk+1)));
end


## 五、差分方程式的解法 (2.6) —— 例 2.10：逐次代入法

求解 $m(k)=e(k)-e(k-1)-m(k-1),\ k\ge0$，其中 $e(k)$ 在偶數 $k$ 為 1、奇數 $k$ 為 0，$e(-1)=m(-1)=0$。

這是數位電腦求解差分方程式最直接的方式：從 $k=0$ 開始一步步往後代入。

In [47]:
mkminus1 = 0;   % m(k-1) 的初始值 m(-1)
ekminus1 = 0;   % e(k-1) 的初始值 e(-1)
ek = 1;         % e(0)
for kk = 0:6
    mk = ek - ekminus1 - mkminus1;
    fprintf('k=%d, m(k)=%d\n', kk, mk);
    mkminus1 = mk;       % 這一輪算出的新值，變成下一輪的「舊值」
    ekminus1 = ek;
    ek = 1 - ek;          % e(k) 在 0 與 1 之間交替
end

k=0, m(k)=1
k=1, m(k)=-2
k=2, m(k)=3
k=3, m(k)=-4
k=4, m(k)=5
k=5, m(k)=-6
k=6, m(k)=7


**函數介紹：`for k = a:b ... end`**——迴圈語法，讓 `k` 依序取 `a, a+1, ..., b`。這裡完美對應差分方程式「逐次代入」的數學過程：每一次迭代就是計算下一個時間點的 $m(k)$。

**數學 ↔ MATLAB 對照**：

- `mkminus1` ↔ $m(k-1)$；`mk` ↔ $m(k)$
- 迴圈本體最後三行賦值 ↔ 把「這一輪算出的新值」變成「下一輪的舊值」，即時間往前推進一步

z 轉換法也能解出相同的結果（見 `chp2.md` 例 2.11），兩種方法互相驗證。

---

## 六、反 z 轉換 (2.7)

四種方法：**冪級數法（長除法）**、**部分分式展開法**、**反演公式法（留數定理）**、**離散摺積法**。這裡示範部分分式法（含重根）與離散摺積法的 MATLAB 實作。

### 部分分式展開法：`iztrans()`

核心技巧是先展開 $\dfrac{E(z)}{z}$（因為查表用的轉換式分子都帶一個 $z$，如 $\dfrac{z}{z-a}\to a^k$），再乘回一個 $z$。

In [ ]:
syms kk_sym
Ez1 = z/((z-1)*(z-2));
iztrans(Ez1, kk_sym)
% 理論結果：e(k) = 2^k - 1


In [ ]:
%%sym
iztrans(Ez1, kk_sym)


**函數介紹：`iztrans(expr, k)`**——`ztrans()` 的反運算，對符號表達式 `expr`（以 `z` 為自變數）計算反 z 轉換，結果以 `k` 為自變數表示。這一行程式相當於手算部分分式＋查表的整個流程。

> ⚠️ **Octave 的 symbolic 套件沒有 `iztrans`**（到 3.2.2 都沒有，`grep` 整個套件目錄
> 連這個字串都找不到）。本資料夾的 `iztrans.m` 用**留數法**自己補上：
> $e(k)=\sum_{	ext{極點}\,p}\operatorname{Res}\left[E(z)z^{k-1}
ight]_{z=p}$，
> 單根、重根、共軛複根皆可。引數慣例與 MATLAB 相同，所以這一行不用改寫。


In [ ]:
% 例 2.16：重根情形
Ez2 = z/(z-1)^2;
iztrans(Ez2, kk_sym)
% 理論結果：e(k) = k（$z=1$ 處為二重極點，反演公式需要對留數多做一次微分，見 chp2.md）


In [ ]:
%%sym
iztrans(Ez2, kk_sym)


In [ ]:
% 例 2.14：共軛複數極點 -> residue 找留數與極點，再轉成正弦形式
num14 = [0, 0, -3.894];
den14 = [1, 0, 0.6065];
[r14, p14, k14] = residue(num14, den14);
r14, p14
% 理論結果：p = ±j0.7788（純虛數極點，代表無衰減的振盪）
%           r = ±j2.5001（對應例 2.14 中 k1 = 2.5∠90°）
% 由此可推出 y(k) = -5 * exp(-0.25k) * sin(pi*k/2)（推導見 chp2.md 第六節）

In [ ]:
%%sym
r14
p14


### 離散摺積法：`conv()`

若 $E(z)=E_1(z)E_2(z)$，兩個數列的**離散摺積和** $e(k)=\sum_{n=0}^k e_1(n)e_2(k-n)$ 就是反 z 轉換的結果。`conv()` 這個函數同時扮演「多項式相乘」與「離散摺積」兩個角色——這不是巧合，因為兩者在數學結構上完全相同，這正是 z 轉換把「摺積」變成「乘法」這個核心性質的體現。

In [ ]:
% 例 2.17
e1 = [1 1 1 1 1 1];
e2 = [0 1 2 4 8 16];
e_conv = conv(e1, e2)
% e(3) 應為 7，與例 2.13 用部分分式法得到的 e(k)=-1+2^k 在 k=3 的值 (-1+8=7) 一致

In [ ]:
%%sym
e_conv


---

## 七、模擬圖與訊號流程圖 (2.8)

離散系統的基本元件是**時間延遲**（延遲一個取樣週期 $T$），其轉移函數為 $z^{-1}$，角色對應類比系統中的**積分器**（轉移函數 $s^{-1}$）。$n$ 階系統若逐項照差分方程式畫模擬圖，需要 $2n$ 個延遲元件（**非最小實現**）；但理論上只需要 $n$ 個延遲元件（狀態）就能完整描述系統（**最小實現**）。這個「如何用最少延遲元件表示系統」的問題，正是引出下一節**狀態變數**方法的動機——本節沒有獨立的程式碼範例，重點在觀念的建立。

---

## 八、狀態變數 (2.9) —— 控制標準型 (CCF)

給定轉移函數：

$$
G(z) = \frac{Y(z)}{U(z)} = \frac{b_{n-1}z^{n-1}+\cdots+b_1z+b_0}{z^n+a_{n-1}z^{n-1}+\cdots+a_1z+a_0}
$$

**控制標準型**（Control Canonical Form）讓我們可以「看一眼轉移函數就直接寫出狀態方程式」：

$$
x(k+1) = \begin{bmatrix}0&1&0&\cdots&0\\0&0&1&\cdots&0\\\vdots&&&\ddots&\vdots\\-a_0&-a_1&-a_2&\cdots&-a_{n-1}\end{bmatrix}x(k) + \begin{bmatrix}0\\0\\\vdots\\1\end{bmatrix}u(k), \qquad y(k) = \begin{bmatrix}b_0&b_1&\cdots&b_{n-1}\end{bmatrix}x(k)
$$

**例 2.19**：$G(z) = \dfrac{z^2+2z+1}{z^3+2z^2+z+0.5}$，即 $b_0=1,b_1=2,b_2=1$，$a_0=0.5,a_1=1,a_2=2$。

In [ ]:
% 例 2.19：控制標準型狀態矩陣
b = [1 2 1];      % b0 b1 b2
a0 = 0.5; a1 = 1; a2 = 2;

A_ccf = [0 1 0; 0 0 1; -a0 -a1 -a2]
B_ccf = [0; 0; 1]
C_ccf = b          % [b0 b1 b2]
D_ccf = 0;

In [ ]:
%%sym
A_ccf
B_ccf
C_ccf


**觀察 $A$ 矩陣結構的規律**：最後一列放 $-a_0,-a_1,\dots,-a_{n-1}$（分母係數取負號），其餘位置是移位矩陣（次對角線全 1）；$C$ 向量直接就是分子係數 $[b_0,b_1,\dots,b_{n-1}]$。這個規律讓我們不需要重新推導，直接「照抄」係數即可建立狀態模型。

---

## 九、相似轉換與對角化 (2.10)

**相似轉換**：引入可逆矩陣 $P$，令 $x(k)=Pw(k)$，可得到另一組等價的狀態模型：

$$
A_w = P^{-1}AP, \qquad B_w = P^{-1}B, \qquad C_w = CP, \qquad D_w = D
$$

**關鍵性質**：相似轉換不改變特徵值、行列式、跡（trace），也不改變轉移函數——因為這些都是系統的「本質特性」，不會因為換了一組內部座標而改變。

### 例 2.23：任選 $P$ 做相似轉換

In [ ]:
A = [0.8 1; 0 0.9];
B = [0; 1];
C = [1 0];

P = [1 -1; 1 1];
Aw = inv(P)*A*P
Bw = inv(P)*B
Cw = C*P

In [ ]:
%%sym
Aw
Bw
Cw


**函數介紹：`inv(M)`**——計算方陣 `M` 的反矩陣 $M^{-1}$，是相似轉換公式 $A_w=P^{-1}AP$ 中 $P^{-1}$ 的數值計算。

**數學 ↔ MATLAB 對照**：$A_w=P^{-1}AP$ ↔ `Aw = inv(P)*A*P`（矩陣乘法用 `*`，順序不可顛倒，因為矩陣乘法沒有交換律）。

In [ ]:
disp('驗證特徵值不變：');
eig(A)
eig(Aw)
% 兩者應完全相同：z1=0.8, z2=0.9

In [ ]:
%%sym
eig(A)
eig(Aw)


**函數介紹：`eig(A)`**（單一輸出）——只回傳矩陣 `A` 的特徵值（不含特徵向量），對應特徵方程式 $|zI-A|=0$ 的根。

### 例 2.25：用 `[V,D]=eig(A)` 做對角化

若特徵值相異，取相似轉換矩陣 $P=M$（特徵向量組成的模態矩陣），可以得到對角化的狀態模型 $\Lambda=M^{-1}AM$。

In [ ]:
[M, LAMBDA] = eig(A)

In [ ]:
%%sym
M
LAMBDA


**函數介紹：`[V, D] = eig(A)`**（雙輸出）——計算方陣 `A` 的特徵向量與特徵值。`V` 的每一行是一個特徵向量，`D` 是以特徵值為對角元素的對角矩陣，滿足 $A\cdot V = V\cdot D$。

In [ ]:
disp('驗證 A*M - M*LAMBDA 應接近零矩陣：');
A*M - M*LAMBDA

In [ ]:
%%sym
A*M - M*LAMBDA


**驗證方式**：`A*V - V*D` 若（數值上）接近零矩陣，就代表 `eig()` 的結果滿足特徵值方程式 $AM=M\Lambda$。

> **注意**：MATLAB/Octave 的 `eig()` 回傳的特徵向量通常會做單位化（長度歸一），數值上可能和手算「任取比例常數=1」的結果差一個縮放係數，但代表的是同一個特徵方向，不影響對角化後的特徵值 $\Lambda$ 本身。

---

## 十、由狀態方程式求轉移函數 (2.11)

公式：

$$
G(z) = C[zI-A]^{-1}B+D
$$

推導自對狀態方程式取 z 轉換、解出 $X(z)$、代入輸出方程式（詳見 `chp2.md` 第十節）。MATLAB 中最直接的做法是用 `ss2tf()`。

### 例 2.28

In [ ]:
A28 = [1.35 0.55; -0.45 0.35];
B28 = [0.5; 0.5];
C28 = [1 -1];
D28 = 0;
T28 = 1;

[num28, den28] = ss2tf(A28, B28, C28, D28);
Gz28 = tf(num28, den28, T28)
% 理論結果：1/(z^2-1.7z+0.72)

In [ ]:
%%sym
Gz28


**函數介紹：`ss2tf(A,B,C,D)`**——直接把狀態空間矩陣 $(A,B,C,D)$ 轉換成轉移函數的分子、分母係數向量，內部原理正是 $G(z)=C[zI-A]^{-1}B+D$ 的數值化實作。

**數學 ↔ MATLAB 對照**：

| 數學 | MATLAB |
|---|---|
| $zI-A$ | 內部隱含於 `ss2tf` |
| $[zI-A]^{-1}$ | 內部隱含於 `ss2tf` |
| $C[zI-A]^{-1}B+D$ | `[num,den] = ss2tf(A,B,C,D)` |

也可以用符號運算 `syms z; Gz = C*inv(z*eye(2)-A)*B+D; simplify(Gz)` 手動走一遍公式（見 `chp2.md`），兩者結果一致，但 `ss2tf` 更快速、適合放進程式流程中。

---

> ⚠️ **`ss2tf` 在 Octave 屬於 `signal` 套件**，不在 `control` 裡（MATLAB 則放在
> Control System Toolbox）。少了 `pkg load signal` 會出現 `error: 'ss2tf' undefined`。


## 十一、狀態方程式的解 (2.12)

一般解：

$$
x(k) = \Phi(k)x(0) + \sum_{j=0}^{k-1}\Phi(k-1-j)Bu(j), \qquad \Phi(k) \triangleq A^k
$$

$\Phi(k)$ 稱為**狀態轉移矩陣**（state transition matrix）。可以用**遞迴法**（電腦逐步代入，例 2.29）求數值解，或用 **z 轉換法**（$\Phi(k)=\mathcal{Z}^{-1}[z[zI-A]^{-1}]$）求 $\Phi(k)$ 的封閉公式（例 2.30，見 `chp2.md`）。

### 例 2.29：遞迴解

In [ ]:
A29 = [0 1; -2 -3];
B29 = [0; 1];
C29 = [3 1];
x = [0; 0];
u = 1;
for kk = 0:5
    x1 = A29*x + B29*u;
    y = C29*x;
    fprintf('k=%d, y(k)=%g\n', kk, y);
    x = x1;   % 這次算出的新狀態，變成下一輪迭代的「目前狀態」
end
% 理論結果：y = 0 1 1 -1 5 -9

**數學 ↔ MATLAB 對照**：迴圈內 `x1 = A29*x + B29*u` 正是遞迴公式 $x(k+1)=Ax(k)+Bu(k)$ 的直接數值實作，完全不需要事先求出 $\Phi(k)$ 的封閉公式。

### 例 2.31：狀態轉移矩陣的性質

$$
\Phi(0)=I, \qquad \Phi(k_1+k_2)=\Phi(k_1)\Phi(k_2), \qquad \Phi(-k)=\Phi^{-1}(k)
$$

In [ ]:
A31 = [1 0; 0 0.5];
Phi = @(kk) A31^kk;    % 注意：^ 是矩陣次方，不是逐元素次方 .^

disp('Phi(0) 應為單位矩陣：');
Phi(0)

In [ ]:
%%sym
Phi(0)


**⚠️ 特別注意**：`A^k` 代表矩陣 $A$ 自乘 $k$ 次（矩陣次方）；若誤寫成 `A.^k`（逐元素次方），MATLAB/Octave 會把矩陣中每個元素各自獨立做 $k$ 次方，結果完全不同，也不再具有 $\Phi(k)=A^k$ 的物理意義。

In [ ]:
k1 = 2; k2 = 3;
disp('Phi(k1+k2) - Phi(k1)*Phi(k2) 應接近零矩陣：');
Phi(k1+k2) - Phi(k1)*Phi(k2)

In [ ]:
%%sym
Phi(k1+k2) - Phi(k1)*Phi(k2)


驗證通過（結果為零矩陣），代表匿名函數 `Phi` 確實滿足狀態轉移矩陣的指數律性質 $\Phi(k_1+k_2)=\Phi(k_1)\Phi(k_2)$。

**函數介紹：`@(kk) A31^kk`**——MATLAB/Octave 的**匿名函數**（anonymous function）語法，`@(參數) 表達式` 定義一個以 `kk` 為輸入、回傳 `A31^kk` 的函數，可以像一般函數一樣呼叫（如 `Phi(0)`、`Phi(5)`），方便重複計算不同 $k$ 值的 $\Phi(k)=A^k$。

---

## 十二、線性時變系統 (2.13)

若系統矩陣 $A(k),B(k),C(k),D(k)$ 隨時間變化，狀態轉移矩陣需要改為：

$$
\Phi(k,k_0) = A(k-1)A(k-2)\cdots A(k_0) = \prod_{j=k_0}^{k-1}A(j)
$$

必須在每個時間點重新計算，不能像非時變系統一樣直接套用 $A^k$。當 $A$ 不再是 $k$ 的函數時，$\Phi(k,k_0)=A^{k-k_0}$，恰好化簡回非時變系統的公式——**非時變系統只是時變系統的特例**。本節純屬觀念延伸，教材未提供獨立數值範例。

---

## 十三、本章總結

本 Notebook 完整走過了：z 轉換的定義與性質、由 s 域函數求 z 轉換、差分方程式的三種解法、反 z 轉換的四種方法、模擬圖與訊號流程圖、狀態變數模型的建立（控制標準型）、相似轉換與對角化、由狀態方程式反推轉移函數、以及狀態方程式的解（狀態轉移矩陣）。這些工具是後續章節（取樣重建、開迴路/閉迴路分析、穩定性、數位控制器設計）的數學基礎。

完整理論推導與所有訂正說明請見 `chp2.md`；純程式碼精簡版請見 `chp2.m`。